# bRAG: Basic (naive) RAG Implementation

This notebook demonstrates a complete implementation of a basic RAG system that enables question-answering over PDF documents. 
The implementation is model, database, and document loader agnostic, though it's currently configured with:
- LLM: DeepSeek (OpenAI-compatible API)
- Vector Database: Pinecone
- Document Loader: PyPDFLoader

The system combines several key components:
1. Document Loading: Loads PDF documents (extensible to other document types)
2. Text Processing: Splits documents into manageable chunks
3. Vector Operations:
   - Embeds text using HuggingFace sentence-transformers model
   - Stores vectors in Pinecone vector database
4. Retrieval System: Implements efficient document retrieval
5. LLM Integration: Uses DeepSeek model for generating responses

All components can be swapped out for alternatives (e.g., different LLMs, vector stores, or document loaders) 
while maintaining the same overall architecture.

This implementation serves as a foundation for building more complex RAG applications
and can be customized based on specific use cases.

----------------------------------------

## Pre-requisites (optional but recommended)

### Only do the first step if you have never created a virtual environment for this repository. Otherwise, make sure that the Python Kernel that you selected is from your `venv/` folder.

In [ ]:
# Create virtual environment
! python -m venv venv

In [ ]:
# Activate virtual Python environment
! source venv/bin/activate

In [ ]:
# If your Python is not from your venv path, ensure that your IDE's kernel selection (on the top right corner) is set to the correct path 
# (your path output should contain "...venv/bin/python")

! which python

In [12]:
# Install all packages
! pip install -r requirements.txt --quiet

## Environment

`(1) Packages`

In [ ]:
import os
# python-dotenv 是一个常用的工具，它允许你从 .env 文件中读取环境变量，这对于管理配置信息（如 API 密钥、数据库连接字符串等）非常有用，特别是当你不想将这些敏感信息直接硬编码在代码中时。
# from dotenv import load_dotenv

# Load all environment variables from .env file
# load_dotenv()

# Access the environment variables
langchain_tracing_v2 = os.getenv('LANGCHAIN_TRACING_V2')
langchain_endpoint = os.getenv('LANGCHAIN_ENDPOINT')
langchain_api_key = os.getenv('LANGCHAIN_API_KEY')

## LLM - DeepSeek (OpenAI-compatible)
api_key = os.getenv('DEEPSEEK_API_KEY').strip()
base_url = os.getenv('DEEPSEEK_API_BASE').strip()

## Pinecone Vector Database
pinecone_api_key = os.getenv('PINECONE_API_KEY')
pinecone_api_host = os.getenv('PINECONE_API_HOST')
index_name = os.getenv('PINECONE_INDEX_NAME')


api_key--- sk-4cede764d3d14a1887a23bad816a7659
base_url--- https://api.deepseek.com


`(2) LangSmith`

https://docs.smith.langchain.com/

In [ ]:
os.environ['LANGCHAIN_TRACING_V2'] = langchain_tracing_v2
os.environ['LANGCHAIN_ENDPOINT'] = langchain_endpoint
os.environ['LANGCHAIN_API_KEY'] = langchain_api_key

`(3) API Keys`

In [2]:
# DeepSeek 使用 OpenAI 兼容接口，通过 base_url 指向 DeepSeek 服务
os.environ['OPENAI_API_KEY'] = api_key
os.environ['OPENAI_API_BASE'] = base_url
deepseek_model = "deepseek-chat"

# Pinecone keys
os.environ['PINECONE_API_KEY'] = pinecone_api_key
os.environ['PINECONE_API_HOST'] = pinecone_api_host
os.environ['PINECONE_INDEX_NAME'] = index_name

`(4) Pinecone Init`

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(os.environ['PINECONE_INDEX_NAME'])

## Full RAG App (Basic)

In [ ]:
# 这段代码实现了一个 RAG（检索增强生成） 流程，用于从 PDF 文档中提取信息并基于 LLM（此处使用 DeepSeek 模型）回答问题。
from langchain_community.document_loaders import PyPDFLoader #  用于加载PDF文档
from langchain.text_splitter import RecursiveCharacterTextSplitter #  用于递归分割文本
# PineconeVectorStore 是一个连接器，让你能通过 LangChain 操作 Pinecone。
# Pinecone 是一个云端的、完全托管的向量数据库。
# 你不需要在本地安装数据库，只需注册账号获取 API 密钥，通过网络调用即可。
# 开发和测试时，可以通过 Docker 使用官方模拟器 Pinecone Local。
from langchain_pinecone import PineconeVectorStore #  用于Pinecone向量存储
from langchain_core.output_parsers import StrOutputParser #  用于解析输出为字符串
# RunnablePassthrough认识
from langchain_core.runnables import RunnablePassthrough #  用于传递运行时数据
from langchain_openai import ChatOpenAI #  用于OpenAI聊天模型
# HuggingFaceEmbeddings认识：包装器，让你能在 LangChain 框架内，便捷地使用 Hugging Face 上成千上万的本地文本嵌入模型
from langchain_huggingface import HuggingFaceEmbeddings #  用于HuggingFace嵌入模型
from langchain.prompts import ChatPromptTemplate #  用于聊天提示模板

#### 1.索引部分，用于文档处理和索引创建
pdf_file_path = "test/langchain_turing.pdf" #  定义PDF文件路径
loader = PyPDFLoader(pdf_file_path) #  创建PDF加载器实例
# 语法：创建 PyPDFLoader 实例，调用 load() 方法返回文档列表（每个元素是一个 Document 对象，包含 page_content 和 metadata）
docs = loader.load() #  加载PDF文档内容

# 2.Split 分割文本
# 作用：将长文档按 1000 字符分块，块间重叠 200 字符，保持语义连贯。
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 语法：split_documents() 接收文档列表，返回分割后的文档块列表。
splits = text_splitter.split_documents(docs) #  使用文本分割器将文档分割成更小的块

# 3.使用 HuggingFace 本地 embedding 模型，无需 API key，该模型将文本转换为向量
#  创建一个 HuggingFaceEmbeddings 实例，使用指定的模型名称
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # 指定 Hugging Face 模型库中的模型 ID 或本地路径。
    # model_kwargs={'device': 'cuda'}, # 强制使用 GPU 加速;或者'device': 'cpu'，强制使用 CPU
    # encode_kwargs={"normalize_embeddings": True}, # 对生成的向量进行 L2 归一化，对使用余弦相似度计算的任务很重要
) 
# all-MiniLM-L6-v2 是一个非常经典和优秀的选择，向量维度：384，特点：平衡之选。速度快、占用资源少，性能均衡，是大多数任务的绝佳起点。

# 4 创建 Pinecone 向量存储
# 将文档分割块转换为向量并存储到 Pinecone 数据库中
vectorstore = PineconeVectorStore.from_documents( #  使用分割后的文档和 embedding 模型创建 Pinecone 向量存储 将文档分割块转换为向量并存储到 Pinecone 数据库中
    documents=splits,  #  分割后的文档块
    embedding=embedding_model,  #  用于生成向量嵌入的模型
    index_name=index_name #  Pinecone 索引名称
)

# 5 创建检索器
retriever = vectorstore.as_retriever() #  创建检索器，用于后续从向量存储中检索相关文档


# 3. 检索与生成部分（RETRIEVAL and GENERATION）
# 3.1 提示模板（Prompt Template）
#  定义一个模板字符串，用于构建提示 这个模板要求基于提供的上下文来回答问题
template = """Answer the question based only on the following context: 
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template) #  使用模板创建聊天提示模板

# 3.2 LLM 配置（使用 DeepSeek）
# LLM - 使用 DeepSeek，通过 ChatOpenAI 的 openai_api_base 指向 DeepSeek 接口
llm = ChatOpenAI( 
    model_name=deepseek_model, #  指定使用的模型名称
    temperature=0.1, #  设置温度参数，控制输出的随机性，值越小输出越确定
    openai_api_key=api_key, #  设置OpenAI API的密钥
    openai_api_base=base_url #  设置OpenAI API的基础URL
)

# Post-processing
# 3.3 后处理函数（Post-processing）
# 语法：普通函数，接收文档列表，提取每个文档的 page_content 并用两个换行连接，用于将检索结果拼接到提示模板的 {context} 位置。
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3.4 RAG 链（Chain）
# Chain #  创建一个RAG（检索增强生成）链，这是一个处理文档检索和问答的完整流程 这个链将检索相关文档，格式化文档内容，然后生成回答

# 语法亮点：使用 LangChain 的 LCEL（LangChain Expression Language） 语法，通过管道符 | 串联组件。
# 第一个字典是一个 RunnableParallel 对象：
# "context" 键对应的值是一个管道：retriever 检索文档 → format_docs 格式化。
# "question" 键对应 RunnablePassthrough()，表示直接传递用户输入的问题。
# 然后依次经过 prompt（格式化提示）、llm（生成回答）、StrOutputParser()（将 LLM 的输出消息解析为字符串）。
# 整体效果：rag_chain 是一个可调用对象，输入问题（字符串），输出答案（字符串）。
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# 总结
# 该代码演示了一个完整的 RAG 流程：

# 加载 PDF → 分割文档 → 生成向量并存入 Pinecone。

# 构建提示模板，配置 LLM（通过 OpenAI 兼容接口调用 DeepSeek）。

# 使用 LCEL 将检索器、提示模板、LLM 和输出解析器串联成链。

# 最终 rag_chain 可用来回答基于文档内容的问题。

In [ ]:
# Question
from pprint import pprint

pprint(rag_chain.invoke("What is this document about?"))